# Module 2: DeepEval Evaluation for Multi-Agent System

## Overview

This notebook evaluates the multi-agent e-commerce customer service system using **DeepEval**.

### Evaluation Approach
- **5 diverse test cases** covering order, product, account, multi-agent, and edge cases
- **3 key metrics**: Helpfulness, Goal Success, Routing Accuracy
- **Performance tracking**: Latency and tool call success

### Time: ~15 minutes

## Step 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q deepeval strands-agents strands-agents-tools boto3 mcp pandas

In [ ]:
import os
import sys
import json
import time
import re
import boto3
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field, asdict

# Get AWS region
session = boto3.Session()
REGION = session.region_name or 'us-west-2'
os.environ['AWS_REGION'] = REGION

# Get table names from SSM
ssm = boto3.client('ssm', region_name=REGION)

try:
    os.environ['ORDERS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-orders-table')['Parameter']['Value']
    os.environ['ACCOUNTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-accounts-table')['Parameter']['Value']
    os.environ['PRODUCTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-products-table')['Parameter']['Value']
    print(f"✓ AWS Region: {REGION}")
    print(f"✓ Tables configured")
except Exception as e:
    print(f"Using default table names")
    os.environ['ORDERS_TABLE_NAME'] = 'ecommerce-workshop-orders'
    os.environ['ACCOUNTS_TABLE_NAME'] = 'ecommerce-workshop-accounts'
    os.environ['PRODUCTS_TABLE_NAME'] = 'ecommerce-workshop-products'

In [ ]:
# DeepEval imports
from deepeval import evaluate
from deepeval.evaluate.configs import DisplayConfig, AsyncConfig
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
from deepeval.models import DeepEvalBaseLLM

print("✓ DeepEval imported successfully")

## Step 2: Amazon Bedrock LLM for Evaluation

In [ ]:
class BedrockEvalModel(DeepEvalBaseLLM):
    """Amazon Bedrock LLM wrapper for DeepEval using Claude Haiku."""
    
    def __init__(self, model_id: str = "global.anthropic.claude-haiku-4-5-20251001-v1:0", region: str = None):
        self.model_id = model_id
        self.region = region or os.environ.get('AWS_REGION', 'us-west-2')
        self.client = boto3.client('bedrock-runtime', region_name=self.region)
    
    def load_model(self):
        return self.model_id
    
    def generate(self, prompt: str, **kwargs) -> str:
        import random
        max_retries = 5
        for attempt in range(max_retries):
            try:
                response = self.client.converse(
                    modelId=self.model_id,
                    messages=[{"role": "user", "content": [{"text": prompt}]}],
                    inferenceConfig={"maxTokens": 1024, "temperature": 0.0}
                )
                return response['output']['message']['content'][0]['text']
            except Exception as e:
                if "throttl" in str(e).lower() and attempt < max_retries - 1:
                    time.sleep((2 ** attempt) + random.random())
                else:
                    raise
    
    async def a_generate(self, prompt: str, **kwargs) -> str:
        return self.generate(prompt, **kwargs)
    
    def get_model_name(self) -> str:
        return self.model_id

eval_model = BedrockEvalModel(region=REGION)
print(f"✓ Evaluation Model: {eval_model.model_id}")

## Step 3: Select 5 Diverse Test Cases

In [ ]:
# Load full dataset
with open('evaluation_dataset.json', 'r') as f:
    eval_data = json.load(f)

all_test_cases = eval_data['test_cases']
print(f"Loaded {len(all_test_cases)} total test cases")

# Select 5 diverse test cases from different categories
SELECTED_IDS = [
    'order_001',      # Order domain: order status
    'product_001',    # Product domain: product search  
    'account_003',    # Account domain: membership benefits
    'complex_001',    # Multi-agent: return + recommendation
    'edge_001',       # Edge case: informal language
]

test_cases_data = [tc for tc in all_test_cases if tc['id'] in SELECTED_IDS]

print(f"\nSelected {len(test_cases_data)} diverse test cases:")
for tc in test_cases_data:
    print(f"  - {tc['id']} ({tc['category']}): {tc['input'][:50]}...")

## Step 4: Initialize Multi-Agent System

In [ ]:
# Add module 1 to path
module1_path = os.path.abspath('../01-multi-agent-prototype')
sys.path.insert(0, module1_path)
sys.path.insert(0, os.path.join(module1_path, 'agents'))

from orchestrator import MultiAgentCustomerService

customer_service = MultiAgentCustomerService(region=REGION)
print("✓ Multi-Agent Customer Service System initialized")

## Step 5: Define LLM-Based Tool Success Evaluation

Since tool call information isn't directly available in the response string, we use an LLM to evaluate whether the expected tools were likely used based on the response content. This approach:
- Analyzes if the response contains information that would require using the expected tools
- Returns a score from 0.0 to 1.0
- More reliable than regex-based extraction

In [ ]:
@dataclass
class EvalResult:
    """Container for a single evaluation result."""
    test_id: str
    category: str
    input: str
    response: str
    ground_truth: str
    latency_ms: float
    expected_tools: List[str] = field(default_factory=list)
    tool_success_score: float = 0.0
    error: Optional[str] = None


def evaluate_tool_success_with_llm(
    query: str,
    response: str,
    expected_tools: List[str],
    eval_model: BedrockEvalModel
) -> float:
    """
    Use LLM to evaluate if the expected tools were likely called based on response content.

    Returns a score from 0.0 to 1.0
    """
    if not expected_tools:
        return 1.0

    tool_descriptions = {
        "get_order_status": "retrieves order details like status, items, shipping info",
        "track_shipment": "provides shipment tracking with carrier and delivery date",
        "process_return": "initiates a return request for an order",
        "modify_order": "changes order details like shipping address",
        "get_customer_orders": "lists customer's order history",
        "search_products": "searches for products by keywords or category",
        "get_product_details": "retrieves detailed product specifications",
        "check_inventory": "checks if a product is in stock",
        "get_product_recommendations": "suggests products based on preferences",
        "compare_products": "compares multiple products side by side",
        "get_return_policy": "explains return and warranty policies",
        "get_account_info": "retrieves customer account details",
        "update_shipping_address": "updates customer's shipping address",
        "get_membership_benefits": "explains membership tier benefits",
        "initiate_password_reset": "starts password reset process",
    }

    expected_descriptions = []
    for tool in expected_tools:
        desc = tool_descriptions.get(tool, tool)
        expected_descriptions.append(f"- {tool}: {desc}")

    prompt = f"""Evaluate if the AI assistant's response indicates that the expected tools were used to answer the customer's question.

CUSTOMER QUERY: {query}

ASSISTANT RESPONSE: {response}

EXPECTED TOOLS THAT SHOULD HAVE BEEN USED:
{chr(10).join(expected_descriptions)}

EVALUATION CRITERIA:
- Score 1.0: Response clearly contains information that would require using ALL expected tools
- Score 0.75: Response contains most of the expected information but may be missing some details
- Score 0.5: Response partially addresses the query but is missing significant tool-derived information
- Score 0.25: Response shows minimal evidence of tool usage
- Score 0.0: Response does not contain any information that would come from the expected tools

Respond with ONLY a single number (0.0, 0.25, 0.5, 0.75, or 1.0) representing the tool success score."""

    try:
        result = eval_model.generate(prompt)
        # Extract the score from the response
        score_match = re.search(r'(0\.0|0\.25|0\.5|0\.75|1\.0|0|1)', result.strip())
        if score_match:
            return float(score_match.group(1))
        return 0.0
    except Exception as e:
        print(f"    Warning: Tool evaluation failed: {e}")
        return 0.0


print("✓ Performance metrics defined (using LLM-based tool evaluation)")

## Step 6: Define 3 Key DeepEval Metrics

In [ ]:
# Create 3 focused metrics for faster evaluation

helpfulness_metric = GEval(
    name="Helpfulness",
    criteria="""Evaluate how helpful the response is for the customer:
    1. Does it directly address the question?
    2. Does it provide actionable information?
    3. Is the tone appropriate for customer service?
    Score 0-1 where 1 is excellent.""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.INPUT],
    threshold=0.7,
    model=eval_model
)

goal_success_metric = GEval(
    name="Goal Success",
    criteria="""Evaluate if the agent achieved the customer's goal:
    1. Did the agent understand what the customer wanted?
    2. Was the customer's issue resolved or clearly addressed?
    3. Does the response match the expected outcome?
    Score 0-1 where 1 means goal fully achieved.""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.INPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    threshold=0.7,
    model=eval_model
)

routing_accuracy_metric = GEval(
    name="Routing Accuracy", 
    criteria="""Evaluate if the request was handled by the correct agent type:
    - Order Agent: orders, tracking, returns, cancellations
    - Product Agent: product search, inventory, recommendations
    - Account Agent: account info, membership, passwords
    Score 1.0 if correct domain, 0.5 if partially correct, 0.0 if wrong.""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.INPUT],
    threshold=0.8,
    model=eval_model
)

metrics = [helpfulness_metric, goal_success_metric, routing_accuracy_metric]
print(f"✓ Created {len(metrics)} evaluation metrics")

## Step 7: Run Agent Evaluation

In [ ]:
def run_agent_evaluation(customer_service, test_cases: List[dict], eval_model: BedrockEvalModel) -> List[EvalResult]:
    """Run the multi-agent system on test cases and collect results."""
    results = []
    
    print(f"Running {len(test_cases)} test cases...\n")
    
    for i, tc in enumerate(test_cases):
        test_id = tc['id']
        query = tc['input']
        expected_tools = tc.get('expected_tools', [])
        
        print(f"[{i+1}/{len(test_cases)}] {test_id} ({tc['category']})")
        print(f"    Query: {query[:60]}...")
        
        start_time = time.time()
        
        try:
            response = customer_service.chat(query)
            latency_ms = (time.time() - start_time) * 1000
            
            # Use LLM to evaluate tool success based on response content
            print(f"    Evaluating tool usage with LLM...")
            tool_success_score = evaluate_tool_success_with_llm(
                query, response, expected_tools, eval_model
            )
            
            print(f"    Latency: {latency_ms:.0f}ms")
            print(f"    Expected Tools: {expected_tools}")
            print(f"    Tool Success Score: {tool_success_score:.2f}")
            
            results.append(EvalResult(
                test_id=test_id,
                category=tc['category'],
                input=query,
                response=response,
                ground_truth=tc.get('ground_truth', ''),
                latency_ms=latency_ms,
                expected_tools=expected_tools,
                tool_success_score=tool_success_score
            ))
            
        except Exception as e:
            print(f"    ERROR: {str(e)[:50]}")
            results.append(EvalResult(
                test_id=test_id,
                category=tc['category'],
                input=query,
                response=f"ERROR: {str(e)}",
                ground_truth=tc.get('ground_truth', ''),
                latency_ms=(time.time() - start_time) * 1000,
                expected_tools=expected_tools,
                error=str(e)
            ))
        
        print()
        time.sleep(0.5)  # Rate limiting
    
    return results


# Run evaluation
eval_results = run_agent_evaluation(customer_service, test_cases_data, eval_model)

## Step 8: Calculate Performance Summary

In [ ]:
# Calculate performance metrics
successful_results = [r for r in eval_results if r.error is None]
latencies = [r.latency_ms for r in successful_results]
tool_scores = [r.tool_success_score for r in successful_results]

print("="*60)
print("PERFORMANCE METRICS")
print("="*60)
print(f"\nTest Cases: {len(eval_results)}")
print(f"Successful: {len(successful_results)}")
print(f"Errors: {len(eval_results) - len(successful_results)}")

if latencies:
    import statistics
    print(f"\nLatency:")
    print(f"  Mean: {statistics.mean(latencies)/1000:.1f}s")
    print(f"  Median: {statistics.median(latencies)/1000:.1f}s")
    print(f"  Min: {min(latencies)/1000:.1f}s")
    print(f"  Max: {max(latencies)/1000:.1f}s")

if tool_scores:
    avg_tool_score = statistics.mean(tool_scores)
    status = "PASS" if avg_tool_score >= 0.7 else "FAIL"
    print(f"\nTool Success (LLM-evaluated):")
    print(f"  Mean Score: {avg_tool_score:.2f} [{status}]")
    print(f"  Min: {min(tool_scores):.2f}")
    print(f"  Max: {max(tool_scores):.2f}")

## Step 9: Run DeepEval Quality Metrics

In [ ]:
# Create DeepEval test cases
deepeval_test_cases = []
for r in eval_results:
    if r.error is None:
        tc = LLMTestCase(
            input=r.input,
            actual_output=r.response,
            expected_output=r.ground_truth,
            context=[f"Category: {r.category}"],
        )
        deepeval_test_cases.append(tc)

print(f"✓ Created {len(deepeval_test_cases)} DeepEval test cases")

In [ ]:
# Run DeepEval (3 metrics × 5 test cases = 15 evaluations)
print("Running DeepEval quality metrics...")
print("(3 metrics × 5 test cases = 15 LLM evaluations)\n")

display_config = DisplayConfig(print_results=True, verbose_mode=False)
async_config = AsyncConfig(run_async=True, max_concurrent=3)

deepeval_results = evaluate(
    test_cases=deepeval_test_cases,
    metrics=metrics,
    display_config=display_config,
    async_config=async_config,
)

## Step 10: Quality Metrics Summary

In [ ]:
# Extract and summarize quality metrics
import statistics

metric_scores = {}
for test_result in deepeval_results.test_results:
    for metric_data in test_result.metrics_data:
        name = metric_data.name
        if name not in metric_scores:
            metric_scores[name] = []
        if metric_data.score is not None:
            metric_scores[name].append(metric_data.score)

print("="*60)
print("QUALITY METRICS SUMMARY")
print("="*60)

for name, scores in metric_scores.items():
    if scores:
        mean = statistics.mean(scores)
        status = "✓ PASS" if mean >= 0.7 else "✗ FAIL"
        print(f"\n{name}:")
        print(f"  Mean: {mean:.2f} [{status}]")
        print(f"  Min:  {min(scores):.2f}")
        print(f"  Max:  {max(scores):.2f}")

## Step 11: Results by Test Case

In [ ]:
# Create results table
print("="*80)
print("RESULTS BY TEST CASE")
print("="*80)

for i, (result, tc_result) in enumerate(zip(eval_results, deepeval_results.test_results)):
    tool_status = "PASS" if result.tool_success_score >= 0.7 else "FAIL"
    print(f"\n{result.test_id} ({result.category})")
    print(f"  Latency: {result.latency_ms/1000:.1f}s")
    print(f"  Expected Tools: {result.expected_tools}")
    print(f"  Tool Success Score: {result.tool_success_score:.2f} [{tool_status}]")
    
    for metric_data in tc_result.metrics_data:
        score = metric_data.score if metric_data.score else 0
        status = "PASS" if score >= 0.7 else "FAIL"
        print(f"  {metric_data.name}: {score:.2f} [{status}]")

## Step 12: Save Results

In [ ]:
# Create summary report
report = {
    'timestamp': datetime.now().isoformat(),
    'test_cases': len(eval_results),
    'performance': {
        'latency_mean_s': statistics.mean(latencies)/1000 if latencies else 0,
        'tool_success_mean': statistics.mean(tool_scores) if tool_scores else 0,
    },
    'quality': {name: statistics.mean(scores) for name, scores in metric_scores.items() if scores},
    'details': [
        {
            'test_id': r.test_id,
            'category': r.category,
            'latency_ms': r.latency_ms,
            'expected_tools': r.expected_tools,
            'tool_success_score': r.tool_success_score,
        }
        for r in eval_results
    ]
}

# Save report
with open('deepeval_results.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)
print("✓ Saved results to deepeval_results.json")

## Step 13: Clean Up

In [ ]:
customer_service.cleanup()
print("✓ MCP connections cleaned up")

## Summary

This evaluation covered:

### Test Cases (5 diverse scenarios)
1. **Order Status** - Basic order lookup
2. **Product Search** - Finding products
3. **Membership Benefits** - Account information
4. **Multi-Agent** - Complex query requiring multiple agents
5. **Edge Case** - Informal language handling

### Metrics (3 quality + 1 tool metric)
- **Helpfulness**: Response quality for customer service
- **Goal Success**: Did the agent achieve the customer's goal?
- **Routing Accuracy**: Was the correct agent type used?
- **Tool Success**: LLM-evaluated score based on response content matching expected tool outputs

### Performance
- **Latency**: End-to-end response time
- **Tool Success Score**: LLM judges if response contains expected tool-derived information (0.0-1.0)

### Next Steps
- Review failed test cases to improve prompts
- Use results as baseline for A/B testing in Module 3
- Set up continuous monitoring in Module 4